# La trappola dell'accuratezza: German Credit

**Classe:** quinta SIA · **Durata:** 45 minuti

### Da dove vengono questi dati e cosa non possiamo farci
German Credit: 1000 richieste di credito di una banca tedesca (anni '90), il 70% classificate come "buoni" pagatori. Contiene attributi come stato civile e sesso combinati e "lavoratore straniero": un modello che li usa per decidere un credito pone problemi di discriminazione. **Lo usiamo solo per imparare a valutare un modello, mai per decidere su persone reali.**

*Il file `dati/german_credit.csv` è nel repository (versione UCI, CC BY 4.0, con le etichette di OpenML "credit-g"). Se manca, la prima cella lo scarica da OpenML.*

In [ ]:
import os
import pandas as pd

# Se il notebook è aperto in Colab da GitHub, il CSV non è nella cartella del runtime:
# si legge direttamente dal repository. Chi copia il notebook in un altro repository
# cambia qui utente e nome del repository.
URL_BASE = "https://raw.githubusercontent.com/giovannipedroncelli/repo-demo-formatore/main/ml/dati/"

def leggi_csv(nome):
    percorso = os.path.join("dati", nome)
    if os.path.exists(percorso):
        return pd.read_csv(percorso)
    return pd.read_csv(URL_BASE + nome)

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score

try:
    credito = leggi_csv("german_credit.csv")
except Exception:                      # file assente: lo si scarica da OpenML
    credito = fetch_openml("credit-g", version=1, as_frame=True).frame
credito.shape

**Prevedi.** Quanti clienti 'buoni' e quanti 'cattivi'?

*La mia previsione:* …

In [ ]:
credito["class"].value_counts()

In [ ]:
X = pd.get_dummies(credito.drop(columns="class"))
y = (credito["class"] == "bad").astype(int)   # 1 = cattivo pagatore, la classe che ci interessa trovare
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler().fit(X_train)          # lo scaler impara SOLO dal training set
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

**Prevedi.** Il modello che dice sempre 'buono' che accuratezza avrà? E che recall sui cattivi pagatori?

*La mia previsione:* …

In [ ]:
modelli = {
    "baseline (sempre 'buono')": DummyClassifier(strategy="most_frequent"),
    "regressione logistica": LogisticRegression(max_iter=1000),
    "kNN (k = 5)": KNeighborsClassifier(n_neighbors=5),
    "albero (profondità 4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "SVM": SVC(),
}
for nome, m in modelli.items():
    m.fit(X_train_s, y_train)
    p = m.predict(X_test_s)
    print(f"{nome:<28} accuratezza {accuracy_score(y_test, p):.3f}   recall cattivi {recall_score(y_test, p):.3f}")

**Indaga.** Quale modello batte davvero la baseline? Se fossi la banca, preferiresti un'accuratezza più alta o un recall più alto sui cattivi pagatori? Perché?